# Solution A2 · The components with the power method: convergence and the gap between eigenvalues

**Type**: Implement from scratch

## Objective

Compute the first principal components with the **power method** (repeatedly multiplying a vector by the covariance matrix) and **deflation** (removing the component found), check the equality with `np.linalg.eigh`, and measure that the speed of
convergence depends on the ratio $\lambda_2/\lambda_1$ (the spectral gap).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine

Xw = load_wine().data
wine_C = np.corrcoef(Xw.T)
eig, vec = np.linalg.eigh(wine_C)
eig, vec = eig[::-1], vec[:, ::-1]
narrow_eig = eig.copy(); narrow_eig[1] = 0.98 * eig[0]                    # second eigenvalue almost equal to the first
narrow_C = (vec * narrow_eig) @ vec.T
print("lambda2/lambda1 ratio: wine", round(eig[1] / eig[0], 3), "| narrow", round(narrow_eig[1] / narrow_eig[0], 3))

## Commented solution

In [ ]:
def power(M, tol=1e-12, max_iter=100000, seed=0, true_v=None):
    v = np.random.default_rng(seed).normal(size=len(M))
    v /= np.linalg.norm(v)
    errors = []
    for t in range(1, max_iter + 1):
        w = M @ v
        w /= np.linalg.norm(w)
        if true_v is not None:
            errors.append(1 - abs(w @ true_v))
        converged = 1 - abs(w @ v) < tol                       # the angle with the previous iteration is negligible
        v = w
        if converged:
            break
    return v, v @ M @ v, t, errors


def top_k(M, k):
    M = M.copy()
    eigenvalues, eigenvectors = [], []
    for _ in range(k):
        v, lam, _, _ = power(M)
        eigenvalues.append(lam); eigenvectors.append(v)
        M = M - lam * np.outer(v, v)                          # deflation: I remove the component just found
    return np.array(eigenvalues), np.array(eigenvectors)

In [ ]:
lams, vs = top_k(wine_C, 3)
print("eigenvalues (power):", lams.round(5), "| eigh:", eig[:3].round(5))
_, _, wine_iters, wine_errors = power(wine_C, true_v=vec[:, 0])
_, _, narrow_iters, _ = power(narrow_C, seed=0)
iteration_ratio = narrow_iters / wine_iters
print(f"iterations: wine {wine_iters}, narrow matrix {narrow_iters} ({iteration_ratio:.0f} times more)")

In [ ]:
e = np.array(wine_errors)
e = e[e > 1e-10]
empirical_rate = float(np.median(e[1:] / e[:-1]))
theoretical_rate = (eig[1] / eig[0]) ** 2
print(f"empirical error reduction rate: {empirical_rate:.4f} | (lambda2/lambda1)^2 = {theoretical_rate:.4f}")

## Check

In [ ]:
with verify():
    v, lam, n_it, err = power(wine_C, true_v=vec[:, 0])
    check(abs(v @ vec[:, 0]) > 1 - 1e-10 and close(lam, eig[0]) and len(err) == n_it, "first eigenvector and eigenvalue correct; one error per iteration", "v <- M v / ||M v||")
    lams, vs = top_k(wine_C, 3)
    check(close(lams, eig[:3], rtol=1e-6) and all(abs(vs[i] @ vec[:, i]) > 1 - 1e-8 for i in range(3)), "first three eigenvalues and eigenvectors (up to the sign) match eigh", "deflation: M - lam * outer(v, v)")
    check(wine_iters is not None and wine_iters < 100 and narrow_iters > 20 * wine_iters and close(iteration_ratio, narrow_iters / wine_iters), "wine: fewer than 100 iterations; with the narrow gap over 20 times more", "the convergence rate is (lambda2/lambda1)")
    check(empirical_rate is not None and abs(empirical_rate - theoretical_rate) / theoretical_rate < 0.1 and close(theoretical_rate, (eig[1] / eig[0]) ** 2), "the empirical error rate is (lambda2/lambda1)^2 within 10%", "median of errors[t+1]/errors[t]")
done()

## What you learned

- The power method converges to the principal component by multiplying several times by the matrix: every step amplifies the component along the dominant eigenvector by a factor $\lambda_1/\lambda_2$ with respect to the second. The error in `1 − cos` drops like
  $(\lambda_2/\lambda_1)^{2t}$: with the wines (ratio 0.53) 23 iterations are enough, with a gap of 0.98 you need 513, that is **over twenty times as many**.
- **The gap between eigenvalues controls everything**: if two components have almost equal variances, their direction is ill-defined (a small noise rotates one into the other) and iterative algorithms slow down. It is the reason why components with close eigenvalues must not be
  interpreted one by one.
- **Deflation** lets you compute only the first `k` components without the complete decomposition: it is the basis of the `arpack` and `randomized` solvers that scikit-learn uses when `k` is much smaller than `p` (with `svd_solver="auto"`).
- Numerical error accumulates in deflation (every component is computed from the reduced matrix): for many components block methods are preferred (subspace iteration, Lanczos).

## Common pitfall

Interpreting the second and the third component as "distinct factors" when their eigenvalues are almost equal: PCA cannot tell them apart, and an arbitrary rotation of the two is just as valid.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you compute PCA on a matrix with millions of rows and thousands of columns?</summary>

You don't need the complete decomposition: the first k components are enough. You use iterative or randomized methods (power/subspace iteration, Lanczos, randomized SVD) that work on the matrix with matrix-vector products, or `IncrementalPCA` in blocks if the data don't fit in memory.
The speed depends on the gap between eigenvalues: with small gaps you need more iterations or oversampling.

</details>